In [ ]:
import os 
import json
import requests
import pandas as pd
from mysql import connector
from dotenv import load_dotenv
import http.client

load_dotenv() 

True

In [66]:
API_KEY = os.getenv("API_KEY")
API_HOST = os.getenv("API_HOST")
SEASON = 2024
LEAUGE_ID = 39

In [ ]:
url = "https://api-football-v1.p.rapidapi.com/v3/fixtures/headtohead"

headers = {
	"x-rapidapi-key": API_KEY,
	"x-rapidapi-host": API_HOST
}

querystring = {"leauge":LEAUGE_ID,
               "season": SEASON}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

In [ ]:
import requests

url = "https://v3.football.api-sports.io/standings"

headers = {
    "x-apisports-key": "385cdd06e9c992d556f4931da207b412"
}

params = {
    "league": 39,
    "season": 2024
}

response = requests.get(url, headers=headers, params=params)

# print(response.json())
payload = response.json()

formatted_response = json.dumps(payload, indent=4)
# print(formatted_response)

standings_list  = payload["response"][0]["league"]
formated_standings_list = json.dumps(standings_list, indent=4)
print(f"\n\n\n{formated_standings_list}")


                                            Transform 
                                            parse api response into dataframe

In [ ]:
#rows and c olumn for transforming json reponse to pandas dataframe
rows=[]
column_names = ['season','position','team_id','team','played','won','draw','lost','goals_for','goals_against','goal_diff','points','form']


for club in standings_list:
    season = 2024
    position = club['rank']
    team_id = club['team']['id']
    team=club['team']['name']
    played=club['all']['played']
    won=club['all']['win']
    draw=club['all']['draw']
    lost=club['all']['lose']
    goals_for=club['all']['goals']['for']
    goals_against=club['all']['goals']['against']
    goal_diff=club['goalsDiff']
    points=club['points']
    form=club['form']

tuple_of_club_records= (season,position,team_id,team,played,won,draw,lost,goals_for,goals_against,goal_diff, points, form)

rows.append(tuple_of_club_records)


df=pd.DataFrame(rows, columns=column_names)
print(df.head)

                                    Load

In [ ]:
MYSQL_HOST = os.getenv("MYSQL_HOST")
MYSQL_PORT = os.getenv("MYSQL_PORT")
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE")

In [ ]:
server_conn = connector.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    connection_timeout=30,
    autocommit=False,
    raise_on_warnings=True
)

server_cur = server_conn.cursor()
print(f"Connected to MySQL server!")

[SUCCESS] - Connected to MySQL server!


In [ ]:
server_cur.close()
server_conn.close()

In [ ]:
db_connection = connector.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password= "Kalunda100",
    database = "premier_league_db"
)

cur = db_connection.cursor()
print(f"Connection to database is successful!!!")

[SUCCESS] - Connection to database is successful!!!


In [ ]:
sql_table = "standings"
cur.execute("SHOW TABLES LIKE %s", (f"{sql_table}",))

if cur.fetchone() is None:
    raise SystemExit(f"This table '{sql_table}' is NOT found...please create it...")
else:
    print(f"[SUCCESS] - This table '{sql_table}' exists! Continue to the next phase ! ")

[SUCCESS] - This table 'standings' exists! Continue to the next phase ! 


In [ ]:
#UPSERT operation
table_cols = ['season', 'position', 'team_id', 'team', 'played', 'won', 'draw', 'lost', 'goals_for', 'goals_against', 'goal_diff', 'points', 'form']


standings_df = df[table_cols]
standings_records_tuples = standings_df.itertuples(index=False, name=None)

list_of_standings_records_tuples = list(standings_records_tuples)
UPSERT_SQL = f"""
INSERT INTO {sql_table} 
(season, position, team_id, team, played, won, draw, lost, goals_for, goals_against, goal_diff, points, form)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s) AS src
ON DUPLICATE KEY UPDATE 
position            = src.position,
team                = src.team,
played              = src.played,
won                 = src.won,
draw                = src.draw,
lost                = src.lost,
goals_for           = src.goals_for,
goals_against       = src.goals_against,
goal_diff           =src.goal_diff,
points              =src.points,
form                =src.form;
"""
no_of_rows_uploaded_to_mysql = len(list_of_standings_records_tuples)
try:
    cur.executemany(UPSERT_SQL, list_of_standings_records_tuples)
    db_connection.commit()
    print(f"[SUCCESS] - Upsert attempted for {no_of_rows_uploaded_to_mysql} rows!")
except Exception as e:
    db_connection.rollback()
    print(f"[ERROR] - Rolled back due to this...: {e}")
finally:
    cur.close()
    db_connection.close()
    print("All database connections now closed. \n\nClean up completed.")